# Soccer video pose overlays

Reads rear-view MOV and (if needed) MP4 videos from `sports_proj3/soccer_vids` in OneDrive. Saves an annotated video, frame-by-frame pose CSV, and completion record for each input under `sports_proj3/pose_results`. Run all cells in order. Start with one video, inspect the overlay, then process the remaining files.


## 1 — Install, connect, and configure

In [3]:
%pip -q install -U ultralytics opencv-python-headless "pandas==2.2.3"
#panda 2.2.3 for pse compatibility

Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
print(sys.executable)
print(sys.version)

import torch
print("PyTorch:", torch.__version__)

/Users/emmy/.venvs/soccer-pose/bin/python
3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]
PyTorch: 2.15.0


In [5]:
import json
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

#my specific working directory 
PROJECT_DIR = Path('/Users/emmy/OneDrive - Virginia Tech/sports proj3')
INPUT_DIR = PROJECT_DIR / 'soccer_vids'
OUTPUT_DIR = PROJECT_DIR / 'pose_results'
MODEL_FILE = 'yolo26n-pose.pt'
PROCESSING_VERSION = 'soccer_back_v0' #v0 for final, tried v1 and v2 as tests lol
DETECTION_CONF = 0.25
KEYPOINT_CONF = 0.50
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

#sanity checks for input/output directories and model file
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f'Input folder not found: {INPUT_DIR}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(MODEL_FILE)
print('Input:', INPUT_DIR)
print('Output:', OUTPUT_DIR)
print('Model:', MODEL_FILE, '| Device:', DEVICE)


Input: /Users/emmy/OneDrive - Virginia Tech/sports proj3/soccer_vids
Output: /Users/emmy/OneDrive - Virginia Tech/sports proj3/pose_results
Model: yolo26n-pose.pt | Device: cpu


## 2 — Reusable processing engine (run once)
All model logic lives here. No filenames, athlete names, or dates are hardcoded.

In [6]:

#match each joint name to its index in the YOLO pose model output
JOINTS = {
    "left_shoulder": 5, "right_shoulder": 6,
    "left_hip": 11, "right_hip": 12,
    "left_knee": 13, "right_knee": 14,
    "left_ankle": 15, "right_ankle": 16,
}

#naming specific angle measures of interest 
ANGLES = [f"{side}_{name}" for side in ("left", "right")
          for name in ("hip_angle", "knee_angle", "lower_leg_angle", "trunk_inclination")]


#function to measure different angles, plug in 3 points to get the angle at the middle point
def angle3(a, b, c):
    ab, cb = np.asarray(a, float) - b, np.asarray(c, float) - b
    length = np.linalg.norm(ab) * np.linalg.norm(cb)
    if length < 1e-9:
        return np.nan
    return float(np.degrees(np.arccos(np.clip(np.dot(ab, cb) / length, -1, 1))))

#function to measure inclination of a line segment defined by two points, relative to vertical axis
def inclination(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    if np.linalg.norm(d) < 1e-9:
        return np.nan
    # Unsigned inclination to image vertical; not a 3D joint angle.
    return float(np.degrees(np.arctan2(abs(d[0]), abs(d[1]))))

#picks player closest to screen (largest bounding box) 
def choose_athlete(result, previous_center):
    #check to see if any players were detected, if not return None
    if result.boxes is None or len(result.boxes) == 0:
        return None, previous_center

    boxes = result.boxes.xyxy.cpu().numpy()

    #gets size of box
    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    areas = widths * heights

    #gets index of largest box (which gets the closest player)
    index = int(np.argmax(areas))

    #get coords for the center of the box
    x1, y1, x2, y2 = boxes[index]
    center = np.array([
        (x1 + x2) / 2,
        (y1 + y2) / 2,
    ])

    return index, center

#get selected keypoints and angles for a given frame, store in a dictionary
def pose_row(result, frame, fps, index, camera):
    #store basic info ab frame 
    row = {"frame": frame, "time_seconds": frame/fps, "camera_view": camera,
           "person_detected": index is not None, "selected_person_index": index}

    #coords and confidence for each joint, if detected
    for name in JOINTS:
        row.update({f"{name}_x": np.nan, f"{name}_y": np.nan,
                    f"{name}_confidence": np.nan})

    #make field for each angle of interest, if detected
    for name in ANGLES:
        row[name] = np.nan

    #sanity check if no keypoints
    if index is None or result.keypoints is None or index >= len(result.keypoints.xy):
        return row

    #key points and confidene for player
    xy = result.keypoints.xy[index].cpu().numpy()

    #confidence for each joint 
    conf = (result.keypoints.conf[index].cpu().numpy()
            if result.keypoints.conf is not None else np.zeros(len(xy)))

    #check confidence validity
    valid = lambda *names: all(np.isfinite(conf[JOINTS[k]]) and
                               conf[JOINTS[k]] >= KEYPOINT_CONF for k in names)
    
    #get the x,y coordinates and confidence for each joint, if valid
    for name, k in JOINTS.items():
        row[f"{name}_confidence"] = float(conf[k])
        if valid(name):
            row[f"{name}_x"], row[f"{name}_y"] = map(float, xy[k])

    #calculate measurements for left and right sides 
    for side in ("left", "right"):
        s, h, k, a = [f"{side}_{joint}" for joint in
                       ("shoulder", "hip", "knee", "ankle")]
        if valid(s, h, k):
            row[f"{side}_hip_angle"] = angle3(xy[JOINTS[s]], xy[JOINTS[h]], xy[JOINTS[k]])
        if valid(h, k, a):
            row[f"{side}_knee_angle"] = angle3(xy[JOINTS[h]], xy[JOINTS[k]], xy[JOINTS[a]])
        if valid(k, a):
            row[f"{side}_lower_leg_angle"] = inclination(xy[JOINTS[k]], xy[JOINTS[a]])
    return row

#shows overlay model function
def draw_frame(result, index, row):
    canvas = result.orig_img.copy()
    if index is not None and result.boxes is not None and result.keypoints is not None:
        selected = result.new()
        selected.update(boxes=result.boxes.data[index:index+1].clone(),
                        keypoints=result.keypoints.data[index:index+1].clone())
        canvas = selected.plot(img=canvas, boxes=False, labels=False)
    return canvas

#file paths for output and status 
from pathlib import Path
import shutil

#acceptable video types 
VIDEO_EXTENSIONS = {".mov", ".mp4"}

#make fingerprint for video so we can see if its been processed. 
def source_fingerprint(source):
    info = Path(source).stat()
    return f"{info.st_size}_{info.st_mtime_ns}"


#makes output paths for video, csv, and status json
def output_paths(source, generation):
    source = Path(source).resolve()

    #santiy check for vid type 
    if source.suffix.lower() not in VIDEO_EXTENSIONS:
        raise ValueError(f"Unsupported video: {source.name}")

    directory = OUTPUT_DIR / source.stem

    #locations to save 
    return {
        "video": directory / "annotated.mp4",
        "csv": directory / "pose.csv",
        "status": directory / "status.json",
    }

#function to check if a video has already been processed, and if the outputs are valid
def completed_record(source, generation, paths):
    try:
        record = json.loads(paths["status"].read_text())
        return (
            record.get("status") == "completed"
            and record.get("source") == str(source)
            and record.get("source_generation") == str(generation)
            and record.get("processing_version") == PROCESSING_VERSION
            and all(
                paths[k].is_file() and paths[k].stat().st_size > 0
                for k in ("video", "csv")
            )
        )
    except (OSError, ValueError):
        return False

#finds input vids and checks processing status 
def inventory():
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f"Input folder not found: {INPUT_DIR}")

    rows = []

    #search
    for source in sorted(INPUT_DIR.resolve().rglob("*")):
        if not source.is_file() or source.suffix.lower() not in VIDEO_EXTENSIONS:
            continue

        generation = source_fingerprint(source)
        paths = output_paths(source, generation)

        if completed_record(source, generation, paths):
            status = "completed"
        elif any(path.exists() for path in paths.values()):
            status = "review_required"
        else:
            status = "pending"

        rows.append({
            "source": str(source),
            "generation": generation,
            "status": status,
        })

    return pd.DataFrame(
        rows, columns=["source", "generation", "status"]
    )


#checking to ensure no double completion
def process_video(source, dry_run=True):
    source = Path(source).resolve()
    if not source.is_file():
        raise FileNotFoundError(source)

    generation = source_fingerprint(source)
    paths = output_paths(source, generation)

    if completed_record(source, generation, paths):
        return {"source": str(source), "status": "skipped"}

    if any(path.exists() for path in paths.values()):
        return {
            "source": str(source),
            "status": "review_required",
            "reason": "Partial or nonmatching outputs at target paths",
        }

    camera = "back"

    #initial testing 
    if dry_run:
        return {
            "source": str(source),
            "status": "would_process",
            "camera_view": camera,
        }

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        # Preserve the input extension, including .MOV.
        local_in = tmp / f"input{source.suffix.lower()}"
        shutil.copyfile(source, local_in)

        raw = tmp / "raw.mp4"
        final = tmp / "annotated.mp4"
        csv = tmp / "pose.csv"

        cap = cv2.VideoCapture(str(local_in))
        try:
            if not cap.isOpened():
                raise RuntimeError(f"OpenCV cannot open {source.name}")

            fps = cap.get(cv2.CAP_PROP_FPS)
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            expected_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        finally:
            cap.release()

        if not np.isfinite(fps) or fps <= 0 or width <= 0 or height <= 0:
            raise RuntimeError("Invalid video properties")

        writer = cv2.VideoWriter(
            str(raw),
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (width, height),
        )
        if not writer.isOpened():
            raise RuntimeError("Could not open output writer")

        rows = []
        previous_center = None

        try:
            for i, result in enumerate(model.predict(
                source=str(local_in),
                device=DEVICE,
                conf=DETECTION_CONF,
                stream=True,
                verbose=True,
            )):
                selected, previous_center = choose_athlete(
                    result, previous_center
                )
                row = pose_row(result, i, fps, selected, camera)
                writer.write(draw_frame(result, selected, row))
                rows.append(row)
        finally:
            writer.release()

        if not rows:
            raise RuntimeError("No frames could be processed.")
            raise RuntimeError(
                f"Frame mismatch: got {len(rows)}, expected {expected_frames}"
            )

        pd.DataFrame(rows).to_csv(csv, index=False)

        subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(raw),
            "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
            "-c:v", "libx264", "-preset", "fast",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(final),
        ], check=True)

        if source_fingerprint(source) != generation:
            raise RuntimeError("Input changed during processing")

        paths["video"].parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(final, paths["video"])
        shutil.copyfile(csv, paths["csv"])

        # Write completion record after both outputs are saved.
        record = {
            "status": "completed",
            "source": str(source),
            "source_generation": generation,
            "processing_version": PROCESSING_VERSION,
            "model_file": MODEL_FILE,
            "camera_view": camera,
            "frames": len(rows),
            "output_video": str(paths["video"]),
            "output_csv": str(paths["csv"]),
            "completed_at": datetime.now(timezone.utc).isoformat(),
        }
        paths["status"].write_text(json.dumps(record, indent=2))

    return {
        "source": str(source),
        "status": "completed",
        "frames": len(rows),
        "paths": {key: str(value) for key, value in paths.items()},
    }

#actually running everything
def run_batch(max_videos=2, dry_run=True):
    """Sequential; max_videos counts only non-completed candidates."""
    table=inventory()
    print("Inventory:", table.status.value_counts().to_dict())
    outcomes=[]
    count=0
    for item in table.sort_values("source").itertuples(index=False):
        if item.status == "completed":
            continue
        if max_videos is not None and count >= max_videos:
            break
        count += 1
        if item.status == "review_required":
            outcome={"source":item.source,"status":"review_required"}
        else:
            try:
                outcome=process_video(item.source, dry_run=dry_run)
            except Exception as error:
                outcome={"source":item.source,"status":"failed", "error":str(error)}
        outcomes.append(outcome)
        print(f'{outcome["status"]}: {item.source}')
    print("Batch summary:", pd.Series([x["status"] for x in outcomes]).value_counts().to_dict() if outcomes else "Nothing pending")
    return pd.DataFrame(outcomes)


## 3 — Find input videos

Lists MOV and MP4 inputs and checks completion records.


In [7]:
inventory_df = inventory()
print("Videos:", len(inventory_df))
print(inventory_df["status"].value_counts())
display(inventory_df)

Videos: 35
status
completed    35
Name: count, dtype: int64


,source,generation,status
0,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,84727351_1794078309000000000,completed
1,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,32302086_1794234409000000000,completed
2,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,38039767_1794234416000000000,completed
3,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,33641031_1794234405000000000,completed
4,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,24710393_1794234414000000000,completed
5,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,30328792_1794234481000000000,completed
6,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,28108588_1794234425000000000,completed
7,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,28591593_1794234423000000000,completed
8,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,57570621_1794234447000000000,completed
9,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,43858288_1794234439000000000,completed


## 4 — Create overlays

Start with one video. `DRY_RUN=True` previews inputs without decoding or running the model. `False` processes videos and saves results to Drive. After inspecting one result, set `MAX_VIDEOS=None` to process all remaining videos. Run one batch at a time.


In [8]:
MAX_VIDEOS = None
DRY_RUN = False

batch_df = run_batch(max_videos=MAX_VIDEOS, dry_run=DRY_RUN)
display(batch_df)


Inventory: {'completed': 35}
Batch summary: Nothing pending


""


## 5 — Verify saved results

For completed rows, `paths` contains the saved annotated MP4 and pose CSV locations. Inspect the overlay before running the full batch.


In [9]:
fresh = inventory()
print("Current statuses:", fresh.status.value_counts().to_dict())
display(fresh)

if "batch_df" in globals() and not batch_df.empty:
    display(batch_df)

Current statuses: {'completed': 35}


,source,generation,status
0,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,84727351_1794078309000000000,completed
1,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,32302086_1794234409000000000,completed
2,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,38039767_1794234416000000000,completed
3,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,33641031_1794234405000000000,completed
4,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,24710393_1794234414000000000,completed
5,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,30328792_1794234481000000000,completed
6,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,28108588_1794234425000000000,completed
7,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,28591593_1794234423000000000,completed
8,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,57570621_1794234447000000000,completed
9,/Users/emmy/Library/CloudStorage/OneDrive-Virg...,43858288_1794234439000000000,completed


## Later motion analysis

YOLO estimates 2D keypoint positions and confidence. The CSV stores selected shoulder, hip, knee, and ankle positions, confidence, frame number, and time. Hip/knee/lower-leg angles are derived by this notebook, not directly predicted by YOLO. Trunk inclination is blank for the back view. These are image-plane measurements, not validated 3D joint angles.

The overlay uses the existing heuristic to select one person based on size, image position, and proximity to the previous selection. Check that it follows the intended player when other people appear. Output videos have no audio. Original videos are unchanged.


In [ ]:
from pathlib import Path
import cv2
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# path to get vids
PROJECT_DIR = Path(
    "/Users/emmy/OneDrive - Virginia Tech/sports proj3"
)
INPUT_DIR = PROJECT_DIR / "soccer_vids"

# get videos
videos = sorted(
    path
    for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".mov", ".mp4"}
)

#print all found for sanity 
print(f"Found {len(videos)} videos")

for index, video in enumerate(videos):
    print(f"{index}: {video.name}")

Found 35 videos
0: pk vids  - 1.MOV
1: pk vids  - 10.MOV
2: pk vids  - 11.MOV
3: pk vids  - 12.MOV
4: pk vids  - 13.MOV
5: pk vids  - 14.MOV
6: pk vids  - 15.MOV
7: pk vids  - 16.MOV
8: pk vids  - 17.MOV
9: pk vids  - 18.MOV
10: pk vids  - 19.MOV
11: pk vids  - 2.MOV
12: pk vids  - 20.MOV
13: pk vids  - 21.MOV
14: pk vids  - 22.MOV
15: pk vids  - 23.MOV
16: pk vids  - 24.MOV
17: pk vids  - 25.MOV
18: pk vids  - 26.MOV
19: pk vids  - 27.MOV
20: pk vids  - 28.MOV
21: pk vids  - 29.MOV
22: pk vids  - 3.MOV
23: pk vids  - 30.MOV
24: pk vids  - 31.MOV
25: pk vids  - 32.MOV
26: pk vids  - 33.MOV
27: pk vids  - 34.MOV
28: pk vids  - 35.MOV
29: pk vids  - 4.MOV
30: pk vids  - 5.MOV
31: pk vids  - 6.MOV
32: pk vids  - 7.MOV
33: pk vids  - 8.MOV
34: pk vids  - 9.MOV


In [138]:
import re

# sort by video number: 1, 2, 3, ..., 35
videos = sorted(
    videos,
    key=lambda path: int(re.search(r"(\d+)$", path.stem).group(1))
)

# open the x video
video_path = videos[0]
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError(f"Could not open {video_path.name}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Video:", video_path.name)
print(f"Reported frame rate: {fps:.4f} FPS")
print("Total frames:", total_frames)
print(f"Approximate duration: {total_frames / fps:.2f} seconds")


Video: pk vids  - 1.MOV
Reported frame rate: 59.9750 FPS
Total frames: 480
Approximate duration: 8.00 seconds


In [ ]:
# store the selected contact frame (moment they kick the ball)
contact_frame = None

contact_button = widgets.Button(
    description="Mark ball contact",
    button_style="success"
)

contact_info = widgets.Label(value="Contact has not been marked.")

def mark_contact(button):
    global contact_frame
    contact_frame = slider.value

    contact_info.value = (
        f"Contact marked: frame {contact_frame} | "
        f"Approximate time: {contact_frame / fps:.4f} seconds"
    )

contact_button.on_click(mark_contact)

#apply on screen for ease
display(widgets.VBox([
    contact_button,
    contact_info
]))

#net contact frame(rough estimate of moment ball hits net)
net_contact_frame = None

net_button = widgets.Button(
    description="Mark net contact",
    button_style="info"
)

net_info = widgets.Label(value="Net contact has not been marked.")

def mark_net_contact(button):
    global net_contact_frame

    if contact_frame is None:
        net_info.value = "Mark ball contact first."
        return

    if slider.value <= contact_frame:
        net_info.value = "Net contact must come after ball contact."
        return

    net_contact_frame = slider.value
    elapsed = (net_contact_frame - contact_frame) / fps

    net_info.value = (
        f"Net contact: frame {net_contact_frame} | "
        f"Contact-to-net time: {elapsed:.4f} seconds"
    )

#gets frame and stores on button
net_button.on_click(mark_net_contact)

display(widgets.VBox([net_button, net_info]))
# load this video's frames into memory
cap.release()
reader = cv2.VideoCapture(str(video_path))

frames = []

print("Loading video frames...", flush=True)

try:
    if not reader.isOpened():
        raise RuntimeError(f"Could not open {video_path.name}")

    while True:
        success, frame = reader.read()

        if not success:
            break

        # Resize only the displayed copy
        height, width = frame.shape[:2]
        if width > 900:
            frame = cv2.resize(
                frame,
                (900, round(height * 900 / width))
            )

        success, encoded = cv2.imencode(".jpg", frame)

        if not success:
            raise RuntimeError("Could not encode a frame")

        frames.append(encoded.tobytes())
finally:
    reader.release()

if not frames:
    raise RuntimeError("No frames could be loaded")

print(f"Loaded {len(frames)} frames.", flush=True)

# slide for frame control
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frames) - 1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="850px")
)

#buttons for ease 
previous_button = widgets.Button(description="← Previous frame")
next_button = widgets.Button(description="Next frame →")

image_widget = widgets.Image(format="jpeg", width=900)
info_widget = widgets.Label()

def update_view(change=None):
    frame_number = slider.value
    image_widget.value = frames[frame_number]
    info_widget.value = (
        f"Frame {frame_number} / {len(frames) - 1} | "
        f"Approximate time: {frame_number / fps:.4f} seconds"
    )

def move_previous(button):
    slider.value = max(slider.min, slider.value - 1)

def move_next(button):
    slider.value = min(slider.max, slider.value + 1)

slider.observe(update_view, names="value")
previous_button.on_click(move_previous)
next_button.on_click(move_next)

display(widgets.VBox([
    slider,
    widgets.HBox([previous_button, next_button]),
    info_widget,
    image_widget
]))

update_view()



Loading video frames...
Loaded 162 frames.


In [133]:
player_input = widgets.Text(
    description="Player:",value=video_path.stem,
    placeholder="Enter player name or ID"
)

outcome_input = widgets.Dropdown(
    description="Outcome:",
    options=["Select outcome", "Goal", "Saved", "Missed"]
)

touch_input = widgets.Dropdown(
    description="Prior touch:",
    options=["Select option", "None", "Goalkeeper", "Post/crossbar", "Unclear"]
)

save_button = widgets.Button(
    description="Save kick labels",
    button_style="success"
)

save_info = widgets.Label()

def save_kick_labels(button):
    if contact_frame is None or net_contact_frame is None:
        save_info.value = "Mark ball contact and net contact first."
        return

    if not player_input.value.strip():
        save_info.value = "Enter the player name or ID."
        return

    if outcome_input.value == "Select outcome":
        save_info.value = "Select the outcome."
        return

    if touch_input.value == "Select option":
        save_info.value = "Select whether anything touched the ball before the net."
        return

    row = {
        "video": video_path.name,
        "player": player_input.value.strip(),
        "contact_frame": contact_frame,
        "net_contact_frame": net_contact_frame,
        "reported_fps": fps,
        "contact_to_net_seconds_approx":
            (net_contact_frame - contact_frame) / fps,
        "outcome": outcome_input.value,
        "touch_before_net": touch_input.value,
    }

    labels_path = PROJECT_DIR / "kick_labels.csv"

    if labels_path.exists():
        labels = pd.read_csv(labels_path)

        # Replace this video's previous labels if saving again
        labels = labels.loc[labels["video"] != video_path.name]
    else:
        labels = pd.DataFrame()

    labels = pd.concat(
        [labels, pd.DataFrame([row])],
        ignore_index=True
    )

    labels.to_csv(labels_path, index=False)

    save_info.value = f"Saved {video_path.name} to kick_labels.csv"

display(widgets.VBox([
    player_input,
    outcome_input,
    touch_input,
    save_button,
    save_info
]))

save_button.on_click(save_kick_labels)

In [137]:
import pandas as pd

labels_path = PROJECT_DIR / "kick_labels.csv"
kicks = pd.read_csv(labels_path)

# 'guestimate' ball travel distance
#markings were done manually so we can assume ball traveled 12 yards +- change for frame rate and net impact based on location
DISTANCE_YARDS = 12
DISTANCE_METERS = DISTANCE_YARDS * 0.9144

# find approximate elapsed time from frame labels
kicks["flight_time_seconds"] = (
    (kicks["net_contact_frame"] - kicks["contact_frame"])
    / kicks["reported_fps"]
)

# sanity check: exclude missing, zero, or negative times
valid = (
    kicks["flight_time_seconds"].gt(0)
    & kicks["reported_fps"].gt(0)
)

kicks["estimated_speed_mps"] = float("nan")
kicks.loc[valid, "estimated_speed_mps"] = (
    DISTANCE_METERS / kicks.loc[valid, "flight_time_seconds"]
)

# to other units
kicks["estimated_speed_mph"] = kicks["estimated_speed_mps"] * 2.236936
kicks["estimated_speed_kph"] = kicks["estimated_speed_mps"] * 3.6
kicks["assumed_distance_yards"] = DISTANCE_YARDS

# Flag shots that may have changed speed before the marked endpoint
kicks["direct_flight_candidate"] = (
    valid & kicks["touch_before_net"].eq("None")
)

# save separately, preserving the original labels
speed_path = PROJECT_DIR / "kick_speeds.csv"
kicks.to_csv(speed_path, index=False)

display(
    kicks[[
        "video",
        "outcome",
        "touch_before_net",
        "flight_time_seconds",
        "estimated_speed_mph",
        "estimated_speed_kph",
        "direct_flight_candidate"
    ]].round(2)
)

print("Saved to:", speed_path)

/var/folders/m0/1q_jsr0x4yl2b4zqrkjgslph0000gn/T/ipykernel_80114/4166567781.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  kicks["flight_time_seconds"] = (
/var/folders/m0/1q_jsr0x4yl2b4zqrkjgslph0000gn/T/ipykernel_80114/4166567781.py:

,video,outcome,touch_before_net,flight_time_seconds,estimated_speed_mph,estimated_speed_kph,direct_flight_candidate
0,pk vids - 1.MOV,Goal,NaN,0.55,44.61,71.79,False
1,pk vids - 2.MOV,Goal,NaN,0.52,47.48,76.42,False
2,pk vids - 3.MOV,Goal,NaN,0.48,50.76,81.69,False
3,pk vids - 4.MOV,Goal,NaN,0.67,36.80,59.22,False
4,pk vids - 5.MOV,Goal,NaN,0.58,42.06,67.69,False
5,pk vids - 6.MOV,Goal,NaN,0.52,47.49,76.42,False
6,pk vids - 7.MOV,Goal,NaN,0.56,43.61,70.18,False
7,pk vids - 8.MOV,Goal,NaN,0.60,41.13,66.20,False
8,pk vids - 9.MOV,Missed,Post/crossbar,0.48,51.29,82.55,False
9,pk vids - 10.MOV,Goal,NaN,0.46,53.63,86.30,False


Saved to: /Users/emmy/OneDrive - Virginia Tech/sports proj3/kick_speeds.csv
